In [36]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import torch
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path.cwd().parent  # If notebook is in notebooks/
DATA_DIR = PROJECT_ROOT / "data"
df = pd.read_csv(DATA_DIR / "final_data.csv")

import torch

model = torch.load("../models/mel-vs-nv-full.pth", map_location=device, weights_only=False)
model.eval()
model = model.to(device)

In [37]:
x = torch.randn(1, 3, 224, 224, device=device)

output = model(x)

# print(output.shape)
print(output)

(tensor([0.9858], grad_fn=<MeanBackward1>), tensor([True]))


In [ ]:
target_layer = model.get_submodule("folds.0.encoder.layer4.1.conv2")


print(target_layer)

Module(
  (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
)


In [39]:
# create the Grad CAM Wrapper
import torch.nn as nn


class GradCAMWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)[0].reshape(-1, 1)


cam_model = GradCAMWrapper(model)

In [40]:
# from pytorch_grad_cam import GradCAM
# from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# cam = GradCAM(
#     model=cam_model,
#     target_layers=[target_layer]
# )

# targets = [ClassifierOutputTarget(0)]

In [41]:
from PIL import Image
from pathlib import Path
import numpy as np
import torch

image_path = Path("../data") / df.iloc[0]["path"]

image = Image.open(image_path).convert("RGB")

In [42]:
print(type(image))
print(image.size if hasattr(image, "size") else image.shape)

<class 'PIL.Image.Image'>
(600, 450)


In [43]:
from torchvision import transforms

# preprocess = transforms.Compose([
#     transforms.PILToTensor(),
#     transforms.ConvertImageDtype(torch.float32),
#     transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
# ])

preprocess = transforms.Compose(
    [transforms.PILToTensor(), transforms.ConvertImageDtype(torch.float32), transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))]
)

input_tensor = preprocess(image).unsqueeze(0).to(device)

print(input_tensor.shape)

torch.Size([1, 3, 450, 600])


In [44]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

cam = GradCAM(model=cam_model, target_layers=[target_layer])

In [45]:
called_layers = []


def record_layer(module, inputs, output):
    called_layers.append(module)


handles = []

for name, module in model.named_modules():
    if name.startswith("folds.0.encoder.layer4"):
        handles.append(module.register_forward_hook(lambda m, i, o, name=name: called_layers.append(name)))

with torch.no_grad():
    model(input_tensor)

for handle in handles:
    handle.remove()

print(called_layers)

['folds.0.encoder.layer4.0.conv1', 'folds.0.encoder.layer4.0.bn1', 'folds.0.encoder.layer4.0.relu', 'folds.0.encoder.layer4.0.conv2', 'folds.0.encoder.layer4.0.bn2', 'folds.0.encoder.layer4.0.downsample.0', 'folds.0.encoder.layer4.0.downsample.1', 'folds.0.encoder.layer4.0.relu', 'folds.0.encoder.layer4.1.conv1', 'folds.0.encoder.layer4.1.bn1', 'folds.0.encoder.layer4.1.relu', 'folds.0.encoder.layer4.1.conv2', 'folds.0.encoder.layer4.1.bn2', 'folds.0.encoder.layer4.1.relu']


In [46]:
grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(0)])

grayscale_cam = grayscale_cam[0]

AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam.utils.image import show_cam_on_image

# Original image as RGB, scaled to [0, 1]
rgb_img = np.array(image).astype(np.float32) / 255.0

visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

plt.figure(figsize=(10, 7))
plt.imshow(visualization)
plt.axis("off");